In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

## Import table from Bronze

In [0]:
df=spark.table('workspace.bronze.dwh_sales_details')

In [0]:
df.printSchema()

In [0]:
df.filter((col("sls_order_dt") == 0) | (length(col("sls_order_dt")) != 8)).display()

## **## Clean Date **Fields****

In [0]:
df = (
    df
    .withColumn(
        "sls_order_dt",
        when(
            (col("sls_order_dt") == 0) | (length(col("sls_order_dt")) != 8),
            None
        ).otherwise(to_date(col("sls_order_dt").cast("string"), "yyyyMMdd"))
    )
    .withColumn(
        "sls_ship_dt",
        when(
            (col("sls_ship_dt") == 0) | (length(col("sls_ship_dt")) != 8),
            None
        ).otherwise(to_date(col("sls_ship_dt").cast("string"), "yyyyMMdd"))
    )
    .withColumn(
        "sls_due_dt",
        when(
            (col("sls_due_dt") == 0) | (length(col("sls_due_dt")) != 8),
            None
        ).otherwise(to_date(col("sls_due_dt").cast("string"), "yyyyMMdd"))
    )
)

In [0]:

df.filter(col('sls_order_dt').isNull()).display()

## **## Calculate Sales,Price,Quantity**

In [0]:
df.filter(col('sls_quantity')==0).display()

In [0]:
df=df.withColumn('sls_sales',when(
    (col('sls_sales').isNull())| (col('sls_sales')!=col('sls_quantity')*abs(col('sls_price'))),
    abs(col('sls_price'))*col("sls_quantity")
    ).otherwise(col('sls_sales')))\
    .withColumn(
        "sls_price",
        when(
            (col("sls_price").isNull()) | (col("sls_price") <= 0),
            when(
                col("sls_quantity") != 0,
                col("sls_sales") / col("sls_quantity")
            ).otherwise(0)
        ).otherwise(col("sls_price"))
    )


## **Renaming columns**

In [0]:
rename_columns ={'sls_ord_num':'order_num',
                 'sls_prd_key':'product_key',
                 'sls_cust_id':'customer_id',
                 'sls_order_dt':'order_date',
                 'sls_ship_dt':'ship_date',
                 'sls_due_dt':'due_date',
                 'sls_sales':'sales',
                 'sls_quantity':'quantity',
                 'sls_price':'price'
                }

for old,new in rename_columns.items():
    df=df.withColumnRenamed(old,new)
df.display()


## **Writing to SilverSchema**

In [0]:
df.write.mode('overwrite').format('delta').saveAsTable('workspace.silver.crm_salesdetails')